In [50]:
import sys
import pyomo
import pandas as pd

from pyomo.environ import *
solver = SolverFactory('cbc')


UC

Model, Parameter

In [51]:
from pyomo.environ import*

# =====================
# Model
# =====================
model = ConcreteModel()

# =====================
# Sets
# =====================
model.G = Set(initialize=['G1','G2','G3'])
model.T = Set(initialize=[1,2,3,4,5,6,7,8])

# =====================
# Parameters
# =====================
Pmin = {'G1':20, 'G2':30, 'G3':50}
Pmax = {'G1':150, 'G2':120, 'G3':100}
Cost = {'G1':15, 'G2':15, 'G3':12.35}
StartupCost = {'G1':500, 'G2':400, 'G3':300}
RampUp = {'G1':50, 'G2':50, 'G3':50}
RampDown = {'G1':50, 'G2':100, 'G3':100}

Demand = {1:150, 2:250, 3:220, 4:200, 5:300, 6:160, 7:300, 8:50}

model.Pmin = Param(model.G, initialize=Pmin)
model.Pmax = Param(model.G, initialize=Pmax)
model.Cost = Param(model.G, initialize=Cost)
model.Demand = Param(model.T, initialize=Demand)
model.StartupCost = Param(model.G, initialize=StartupCost)
model.RampUp = Param(model.G, initialize=RampUp)
model.RampDown = Param(model.G, initialize=RampDown)

Variables

In [52]:
# =====================
# Variables
# =====================
model.P = Var(model.G, model.T, domain=NonNegativeReals)
model.u = Var(model.G, model.T, domain=Binary)
model.y = Var(model.G, model.T, domain=Binary)

Objective

In [53]:
# =====================
# Objective
# =====================
def obj_rule(m):
    return sum(m.Cost[g] * m.P[g,t] 
               + m.StartupCost[g] * m.y[g,t]
               for g in m.G for t in m.T)

model.Obj = Objective(rule=obj_rule, sense=minimize)

Constraints

In [54]:
# =====================
# Constraints
# =====================

# Power balance
def balance_rule(m, t):
    return sum(m.P[g,t] for g in m.G) == m.Demand[t]

model.Balance = Constraint(model.T, rule=balance_rule)

# Capacity upper bound
def max_rule(m, g, t):
    return m.P[g,t] <= m.Pmax[g] * m.u[g,t]

model.MaxCap = Constraint(model.G, model.T, rule=max_rule)

# Capacity lower bound
def min_rule(m, g, t):
    return m.P[g,t] >= m.Pmin[g] * m.u[g,t]

model.MinCap = Constraint(model.G, model.T, rule=min_rule)

# Startup Logic
def startup_rule(m, g, t):
    if t == 1:
        return m.y[g,t] >= m.u[g,t]
    else:
        return m.y[g,t] >= m.u[g,t] - m.u[g,t-1]

model.StartupLogic = Constraint(model.G, model.T, rule=startup_rule)

# Ramp Up
def ramp_up_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g,t] - m.P[g,t-1] <= m.RampUp[g]

model.RampUpConstraint = Constraint(model.G, model.T, rule=ramp_up_rule)

# Ramp Down
def ramp_down_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g,t-1] - m.P[g,t] <= m.RampDown[g]

model.RampDownConstraint = Constraint(model.G, model.T, rule=ramp_down_rule)

Solve

In [55]:
# =====================
# Solve
# =====================
solver = SolverFactory('cbc')
results = solver.solve(model)

Results

In [56]:
# =====================
# Format
# =====================
def fmt(x):
    if abs(x - round(x)) < 1e-6:
        return int(round(x))
    else:
        return round(x, 2)
    
# =====================
# Print Results
# =====================
#print("Total System Cost =", fmt(value(model.Obj)))

from pyomo.opt import TerminationCondition
results = solver.solve(model)

if results.solver.termination_condition != TerminationCondition.optimal:
    print("Model infeasible or not optimal.")
else:
    print("Optimal solution found.")

rows = []

for t in model.T:
    g1_p = value(model.P['G1', t])
    g2_p = value(model.P['G2', t])
    g3_p = value(model.P['G3', t])
     # Fuel cost
    g1_fuel = g1_p * value(model.Cost['G1'])
    g2_fuel = g2_p * value(model.Cost['G2'])
    g3_fuel = g3_p * value(model.Cost['G3'])
    
    fuel_cost = g1_fuel + g2_fuel + g3_fuel
    
    # Startup cost
    g1_start = value(model.y['G1', t]) * value(model.StartupCost['G1'])
    g2_start = value(model.y['G2', t]) * value(model.StartupCost['G2'])
    g3_start = value(model.y['G3', t]) * value(model.StartupCost['G3'])
    
    startup_cost = g1_start + g2_start + g3_start
    
    total_cost = fuel_cost + startup_cost
    
    rows.append([
        t,
        fmt(value(model.Demand[t])),
        fmt(g1_p),
        fmt(g2_p),
        fmt(g3_p),
        fmt(fuel_cost),
        fmt(startup_cost),
        fmt(total_cost)
    ])

df_hourly = pd.DataFrame(
    rows,
    columns=[
        "Hour",
        "Demand",
        "G1_P",
        "G2_P",
        "G3_P",
        "Fuel Cost",
        "Startup Cost",
        "Total Cost"
    ]
)

print("\n=== Hourly UC Result ===")
print(df_hourly.to_string(index=False))

# ==========================
# TABEL RAMPING
# ==========================

ramp_rows = []

for t in model.T:
    if t == 1:
        ramp_rows.append([t, "-", "-", "-"])
    else:
        dp1 = value(model.P['G1', t]) - value(model.P['G1', t-1])
        dp2 = value(model.P['G2', t]) - value(model.P['G2', t-1])
        dp3 = value(model.P['G3', t]) - value(model.P['G3', t-1])
        
        ramp_rows.append([
            t,
            fmt(dp1),
            fmt(dp2),
            fmt(dp3)
        ])

df_ramp = pd.DataFrame(
    ramp_rows,
    columns=["Hour", "ΔG1", "ΔG2", "ΔG3"]
)

print("\n=== Ramping Check (ΔP) ===")
print(df_ramp.to_string(index=False))

# ==========================
# SYSTEM SUMMARY
# ==========================

peak_demand = max(value(model.Demand[t]) for t in model.T)
total_cost_system = value(model.Obj)

print("\n=== System Summary ===")
print("Peak Demand =", fmt(peak_demand))
print("Total System Cost =", fmt(total_cost_system))

Optimal solution found.

=== Hourly UC Result ===
 Hour  Demand  G1_P  G2_P  G3_P  Fuel Cost  Startup Cost  Total Cost
    1     150     0    50   100       1985           700        2685
    2     250    50   100   100       3485           500        3985
    3     220    90    30   100       3035             0        3035
    4     200    50    50   100       2735             0        2735
    5     300   100   100   100       4235             0        4235
    6     160    50    50    60       2241             0        2241
    7     300   100   100   100       4235             0        4235
    8      50    50     0     0        750             0         750

=== Ramping Check (ΔP) ===
 Hour ΔG1  ΔG2  ΔG3
    1   -    -    -
    2  50   50    0
    3  40  -70    0
    4 -40   20    0
    5  50   50    0
    6 -50  -50  -40
    7  50   50   40
    8 -50 -100 -100

=== System Summary ===
Peak Demand = 300
Total System Cost = 23901
